# Environmental Economics — When Every Number Says How It Was Made

Every source in this corpus so far publishes a quantity and, separately, something about
how good that quantity is: a margin of error, a suppression flag, a coverage share, a
privacy parameter. The uncertainty statement applies to the dataset, or to a cell, and you
read it once.

The EPA Toxics Release Inventory does something no other source here does. It attaches, to
**every individual release record**, a code saying how that number was produced — continuous
monitoring, periodic monitoring, mass balance, a published emission factor, a site-specific
emission factor, or best engineering judgment. Epistemic provenance arrives per observation.

That is more honest than almost any publisher manages, and this notebook takes it seriously
enough to ask what it actually says. In Virginia's 2022 reporting year, **16 of 2,100
release records rest on continuous monitoring** — three-quarters of one percent. The
largest single category, at 32%, is "other approaches such as engineering calculations or
best engineering judgment."

Then it asks a harder question. Weight those same records by pounds instead of counting them
and the monitored share rises from 12.52% to 39.88% — a much more reassuring number. The
notebook shows that the reassuring number is **one facility**: drop the single largest record
and it falls to 12.22%. Across four states the record-weighted share stays inside a narrow
band and the mass-weighted share does not, and the direction in which that one record moves
it is not even consistent.

The lesson generalises past TRI: *when a quality flag is per-record, the weighting you choose
is a claim about what you are measuring, and a mass-weighted quality statistic over a
concentrated distribution is not a statistic.*

## Learning Objectives

1. **Read a per-record provenance flag** as data about the observation rather than metadata
   about the file, and say what share of a published series rests on measurement.
2. **Choose a weighting deliberately**, and test whether a weighted quality statistic is a
   property of the population or of one observation.
3. **Distinguish a directional loss from a material one**, by bounding a known gap in both
   records and magnitude rather than assuming the sign implies the size.
4. **Detect a vocabulary change inside a single column**, and see why the documented change
   year is not a clean cut in either direction.
5. **Recognise silent truncation** in an API that caps a response and signals nothing, and
   know the only defence available.
6. **Say why a source with no published interval is not a source with no uncertainty.**

## Prerequisites

- Handbook §1.4 (QCEW — suppression with a flag) for the nearest relative: there, the
  publisher marks what is missing. Here the publisher marks how what is present was made.
- Handbook §1.13 (FBI CDE — voluntary reporting) for the other self-report regime. There the
  gap is who filed; here it is how the filer arrived at the number.
- Handbook §7.3 (the closed series on precision and rank) for the contrast this source
  cannot join, and section 8 below for why.
- Comfort with `groupby`, quantile binning, and a leave-one-out sensitivity check.

## Data Sources

**U.S. Environmental Protection Agency, Toxics Release Inventory**, retrieved live from the
Envirofacts REST service at `https://data.epa.gov/efservice`. The query chains three tables:
`tri_facility` (filtered to one state) → `tri_reporting_form` (filtered to one reporting
year) → `tri_release_qty` (one row per environmental medium).

No API key. No registration. The service is public and unauthenticated, which is worth
saying plainly because it is rare among the sources in this corpus.

**What TRI is.** A facility that exceeds a statutory threshold — for employees, for the
quantity of a listed chemical manufactured, processed, or otherwise used — must file a Form R
for each such chemical each year. The facility reports its own releases. EPA does not measure
them and does not verify them. This is a **regulatory self-report**, and the reporting
universe is defined by thresholds, not by a sample design.

**What that means for absence.** A facility that does not appear released nothing *that it
was required to report*. It has not been shown to have released nothing. Two facilities can
be identical in what they emit and differ in whether they appear, because one crossed an
employee count and the other did not.

**Retrieval discipline.** The client in `krl_tutorials_data.sources` calls the Envirofacts
`count` endpoint before it fetches a single row, pages in 2,000-row requests, and refuses to
return a frame whose length disagrees with the declared count. Section 7 shows why.

In [1]:
"""Configuration."""
from __future__ import annotations

import numpy as np
import pandas as pd
from scipy import stats

import sys
from pathlib import Path

#: Find the package by walking UP from wherever this runs, so the notebook works
#: whether it is executed in place or from the repository root. Sixteen notebooks
#: imported `krl_tutorials_data` with no path discovery and therefore ran only
#: from the root -- which nobody noticed while every run happened to start there.
_repo = next((c for c in (Path.cwd().resolve(), *Path.cwd().resolve().parents)
              if (c / "src" / "krl_tutorials_data").is_dir()), None)
if _repo is None:
    raise SystemExit("krl_tutorials_data not found above " + str(Path.cwd()))
if str(_repo) not in sys.path:
    sys.path.insert(0, str(_repo / "src"))

from krl_tutorials_data.sources import (
    verify_digest,
    TRI_BASIS_CODES,
    TRI_CODE_SCHEME_CHANGE_YEAR,
    TRI_MEASURED_CODES,
    TRI_PAGE_LIMIT,
    TRI_RANGE_BANDS,
    TRI_REQUEST_ROWS,
    fetch_tri_releases,
)

#: One state, one reporting year, held fixed. The basis-of-estimate vocabulary
#: changed in 2007 (section 5), so a year is not optional here.
STATE, YEAR = "VA", 2022

#: Three further states, named rather than sampled, to separate a property of
#: the regime from a property of Virginia. Chosen to span size: DE and RI are
#: small, WV is industrial and comparable to VA.
COMPARISON = ["DE", "RI", "WV"]

#: The reporting years used to show the vocabulary boundary. Counts only --
#: no rows are downloaded for these.
SCHEME_YEARS = [2006, 2007, 2008]

pd.set_option("display.width", 110)
print(f"target        : {STATE} reporting year {YEAR}")
print(f"comparison    : {', '.join(COMPARISON)}")
print(f"request size  : {TRI_REQUEST_ROWS:,} rows (API cap is {TRI_PAGE_LIMIT:,})")
print(f"measured codes: {', '.join(sorted(TRI_MEASURED_CODES))}")

#: The digest of the payload every figure below was computed from, on this
#: record's own hash basis. The publisher serves no prior vintage, so this pin
#: does not restore one -- it guarantees an upstream change cannot pass
#: unnoticed: the notebook raises instead of reporting different numbers under
#: unchanged prose. Handbook §5.2.
PINNED_TRI_VA = "33e1653a9a7035e73fbbe0c5af8685a96759cadbc7a586a931c7fb0c1e8fa57f"


target        : VA reporting year 2022
comparison    : DE, RI, WV
request size  : 2,000 rows (API cap is 10,000)
measured codes: M, M1, M2


In [2]:
"""Retrieve the release records."""
releases, prov = verify_digest(
    fetch_tri_releases(
    state=STATE, reporting_year=YEAR, applicable_only=True
),
    PINNED_TRI_VA, label='EPA TRI, Virginia reporting year 2022')

universe = int(prov.variables["universe_rows"])
print(f"medium-rows matched by the query : {universe:,}")
print(f"marked applicable and downloaded : {len(releases):,}")
print(f"filtered server-side             : {universe - len(releases):,}")
print(f"distinct reporting forms         : {releases['doc_ctrl_num'].nunique():,}")
print(f"payload sha256                   : {prov.content_sha256[:16]}...")


medium-rows matched by the query : 12,069
marked applicable and downloaded : 2,100
filtered server-side             : 9,969
distinct reporting forms         : 977
payload sha256                   : 33e1653a9a7035e7...


## 1. The universe, and what a not-applicable row is

A Form R enumerates every environmental medium — stack air, fugitive air, water, several
kinds of land disposal and underground injection — and the filer marks each one applicable or
not. Most are not.

The query for Virginia's 2022 reporting year matches **12,069 medium-rows**, of which
**2,100** are marked applicable (`release_na == '0'`). The other 9,969 carry no basis code and
no quantity, and this notebook leaves them on the server.

That filtering is a download decision, not an accounting one. The client counts the full
universe either way and records both numbers in provenance, because a 2,100-row frame
standing in silently for a 12,069-row query is how a reader loses track of what was excluded.
And the excluded rows are not nothing: a not-applicable row records that the filer
*considered* the medium and reported no release to it. That is different from a medium the
form never asked about.

In [3]:
"""What a not-applicable row is, and why it is not missing data."""
applicable = releases["release_na"].value_counts(dropna=False)
print("release_na among downloaded rows:")
print(applicable.to_string())

with_quantity = releases["total_release"].notna()
print()
print(f"downloaded rows                     : {len(releases):,}")
print(f"  carrying a numeric total_release  : {int(with_quantity.sum()):,}")
print(f"  carrying a range code instead     : {int(releases['release_range_code'].notna().sum()):,}")
print(f"  of the numeric ones, exactly zero : {int((releases['total_release'] == 0).sum()):,}"
      f" ({(releases['total_release'] == 0).sum() / with_quantity.sum():.1%})")
print()
print("environmental media, top 6:")
print(releases["environmental_medium"].value_counts().head(6).to_string())


release_na among downloaded rows:
release_na
0    2100

downloaded rows                     : 2,100
  carrying a numeric total_release  : 2,016
  carrying a range code instead     : 84
  of the numeric ones, exactly zero : 577 (28.6%)

environmental media, top 6:
environmental_medium
AIR STACK    772
AIR FUG      741
WATER        419
OTH DISP      81
OTH LANDF     46
SI 5.5.3B     29


## 2. The basis-of-estimate code

Six codes appear in this slice. Read the table by `record share` first and by `pound share`
second — section 3 is about the distance between them.

The `zero_rate` column is a check on what each code is used *for*: among records carrying a
numeric quantity, the share of those quantities that are exactly zero.

In [4]:
"""The basis-of-estimate code: epistemic provenance, one record at a time."""
by_code = releases.groupby("release_basis_est_code", dropna=False).agg(
    records=("total_release", "size"),
    pounds=("total_release", "sum"),
    median_lb=("total_release", "median"),
    zero_rate=("total_release", lambda s: (s == 0).sum() / s.notna().sum()),
)
by_code["record share"] = by_code["records"] / by_code["records"].sum()
by_code["pound share"] = by_code["pounds"] / by_code["pounds"].sum()
by_code["meaning"] = [TRI_BASIS_CODES[c] for c in by_code.index]
by_code = by_code.sort_values("records", ascending=False)

print(by_code[["records", "record share", "pounds", "pound share",
               "median_lb", "zero_rate"]].to_string(
    formatters={
        "records": "{:,.0f}".format, "record share": "{:.2%}".format,
        "pounds": "{:,.0f}".format, "pound share": "{:.2%}".format,
        "median_lb": "{:,.2f}".format, "zero_rate": "{:.1%}".format,
    }))
print()
for code, row in by_code.iterrows():
    print(f"  {code:<3} {row['meaning']}")


                       records record share     pounds pound share median_lb zero_rate
release_basis_est_code                                                                
O                          672       32.00%    701,172       2.73%      0.00     53.6%
E1                         563       26.81%  6,008,129      23.39%      0.69     17.4%
E2                         303       14.43%  5,248,891      20.43%    258.54      4.5%
C                          299       14.24%  3,485,857      13.57%     60.50     22.4%
M2                         247       11.76% 10,131,468      39.44%     21.00     23.8%
M1                          16        0.76%    115,118       0.45%      0.06     46.2%

  O   other approaches such as engineering calculations or best engineering judgment
  E1  published emission factors (added RY2007)
  E2  site specific emission factors (added RY2007)
  C   mass balance calculations
  M2  periodic or random monitoring data or measurements (added RY2007)
  M1  continuo

Three things in that table are worth naming.

**Continuous monitoring is nearly absent.** `M1` is **16 records, 0.76%** of the slice. Any
of the other five codes is more common.

**The largest category is judgment.** `O` — "other approaches such as engineering
calculations or best engineering judgment" — is **32.00%** of records, more than any single
monitoring or calculation code.

**The codes are not used for the same job.** `O` carries a **53.6%** zero rate and a median
of 0.00 lb: it is heavily the code a filer uses to say *we judge this to be none*. `E2`
(site-specific emission factors) carries a **4.5%** zero rate and a median of 258.54 lb. The
basis code therefore encodes something about the reporting situation as well as the
estimation method, which is a caution against reading it as a pure quality scale. A 46.2%
zero rate on `M1` looks alarming until you notice it is 6 of the 13 `M1` records that carry a
quantity at all — too few to interpret, and stated here so the table does not invite the
interpretation.

## 3. The weighting is a claim

"What share of this series rests on measurement?" has two defensible answers, and they are
not close.

Counting records, **263 of 2,100 = 12.52%**. Weighting by pounds released, **39.88%**. The
second is the more flattering number, and it is the one a reader would reach for if they
cared about total mass rather than about counting filings.

The test below is a leave-one-out over the twenty-five largest records. If 39.88% is a
property of Virginia's industry, dropping any one record should barely move it.

In [5]:
"""Record-weighted and mass-weighted answers to the same question."""
measured = releases["is_measured"].fillna(False)
quantified = releases[releases["total_release"].notna()]
q_measured = quantified["is_measured"].fillna(False)
total_lb = quantified["total_release"].sum()

record_share = measured.mean()
mass_share = quantified.loc[q_measured, "total_release"].sum() / total_lb
print(f"records resting on monitoring : {int(measured.sum()):,} of {len(releases):,}"
      f" = {record_share:.2%}")
print(f"pounds  resting on monitoring : {mass_share:.2%}")
print(f"  of which continuous (M1)    : "
      f"{int((releases['release_basis_est_code'] == 'M1').sum()):,} records"
      f" = {(releases['release_basis_est_code'] == 'M1').mean():.2%}")

# Leave-one-out over the largest records. If the mass-weighted share is a
# property of the state it should barely move; if it is one facility it will.
jackknife = []
for idx in quantified.nlargest(25, "total_release").index:
    rest = quantified.drop(index=idx)
    rest_measured = rest["is_measured"].fillna(False)
    jackknife.append(
        rest.loc[rest_measured, "total_release"].sum() / rest["total_release"].sum()
    )
largest = quantified["total_release"].max()
print()
print(f"largest single record         : {largest:,.0f} lb"
      f" = {largest / total_lb:.2%} of all pounds")
print(f"mass share, dropping only it  : {jackknife[0]:.2%}")
print(f"mass share, leave-one-out over the 25 largest:"
      f" {min(jackknife):.2%} to {max(jackknife):.2%}")
print(f"record share, for comparison  : {record_share:.2%}")


records resting on monitoring : 263 of 2,100 = 12.52%
pounds  resting on monitoring : 39.88%
  of which continuous (M1)    : 16 records = 0.76%

largest single record         : 8,096,003 lb = 31.51% of all pounds
mass share, dropping only it  : 12.22%
mass share, leave-one-out over the 25 largest: 12.22% to 42.22%
record share, for comparison  : 12.52%


It is not a property of Virginia's industry. It is one record.

The single largest release — **8,096,003 lb, 31.51% of every pound in the slice** — is coded
`M2`. Drop that one row and the mass-weighted monitored share falls from **39.88% to 12.22%**,
which is within a third of a percentage point of the record-weighted **12.52%**. Across the
twenty-five largest records the leave-one-out share spans **12.22% to 42.22%**.

A statistic that moves nearly twenty-eight points when one of 2,016 observations is removed
is not describing a population. The record-weighted figure has no such exposure: removing any single
record moves it by at most one part in 2,100.

This is not an argument that record-weighting is always right. It is an argument that the
choice is substantive, that the mass-weighted answer here needs a stability check before it
is quoted, and that the check is four lines long.

## 4. Does measurement follow magnitude?

There is a reasonable prior behind the mass-weighted number: perhaps large releases are the
ones facilities monitor, so weighting by mass legitimately reflects that the important
quantities are better established.

That prior is testable. Bin the positive releases into deciles by size and look at the
monitored share in each.

In [6]:
"""Is measurement concentrated in the large releases, or spread evenly?"""
positive = quantified[quantified["total_release"] > 0].copy()
positive["decile"] = pd.qcut(
    positive["total_release"], 10, labels=False, duplicates="drop"
) + 1
deciles = positive.groupby("decile").agg(
    n=("total_release", "size"),
    low_lb=("total_release", "min"),
    high_lb=("total_release", "max"),
    measured=("is_measured", lambda s: s.fillna(False).mean()),
)
def _lb(value: float) -> str:
    """Pounds span eight orders of magnitude here; one format does not fit."""
    if value == 0:
        return "0"
    return f"{value:,.0f}" if value >= 1 else f"{value:,.4f}"


print(deciles.to_string(formatters={
    "n": "{:,.0f}".format, "low_lb": _lb, "high_lb": _lb,
    "measured": "{:.1%}".format,
}))

rho, pval = stats.spearmanr(
    positive["total_release"], positive["is_measured"].fillna(False).astype(int)
)
print()
print(f"Spearman(release size, is_measured) = {rho:+.4f}  p = {pval:.3f}  n = {len(positive):,}")


         n low_lb   high_lb measured
decile                              
1      144 0.0000    0.0540     4.2%
2      145 0.0583    0.8000    15.2%
3      151 0.8100         5    15.2%
4      137      5        25    10.9%
5      143     25        92    20.3%
6      143     92       311    11.2%
7      144    311       942    13.2%
8      144    964     3,800    14.6%
9      144  3,833    18,540    11.8%
10     144 18,551 8,096,003    14.6%

Spearman(release size, is_measured) = +0.0280  p = 0.288  n = 1,439


The prior does not survive. The monitored share runs between 4.2% and 20.3%
across the ten deciles with no trend in it — decile 5 is the highest and decile 10 is at
14.6%, below decile 2. Spearman's ρ between release size and the monitored flag is
**+0.0280, p = 0.288, n = 1,439**: no detectable relationship.

So the mass-weighted excess in section 3 is not "big releases get measured". It is one
facility that both monitors periodically and happens to account for a third of the state's
reported mass. The two facts are independent and their coincidence is what produced the
reassuring number.

## 5. The range-coded hole

A filer releasing a small quantity may report a **range code** instead of a number. EPA's
Data Dictionary states that when a facility uses a range code, the amount reported to TRI is
the midpoint of that range: code `1` is 1–10 lb, code `3` is 11–499 lb, code `4` is 500–999
lb, and the historical code `2` is 1–499 lb.

The API does not honour that. Every range-coded row in this slice returns `total_release` as
null while `release_na` marks the release **applicable**. Summing `total_release` therefore
drops them silently, and what it drops is specifically the small end — a directional loss,
not noise.

The client reconstructs the documented midpoint into a separate column, `range_midpoint_lb`,
and deliberately does **not** merge it into `total_release`. A reconstruction of what the
documentation says should be stored is not a reported quantity, and merging the two would
hide the distinction this source exists to make visible.

How much does it matter? Bound it in both currencies.

In [7]:
"""The range-coded hole, counted in records and in pounds."""
ranged = releases["release_range_code"].notna()
band_high = releases["release_range_code"].map(
    {code: high for code, (_low, high) in TRI_RANGE_BANDS.items()}
)
midpoint_lb = releases["range_midpoint_lb"].sum()

print("range codes present:")
print(releases.loc[ranged, "release_range_code"].value_counts().sort_index().to_string())
print()
print(f"range-coded records            : {int(ranged.sum()):,} of {len(releases):,}"
      f" = {ranged.mean():.2%}")
print(f"  of those, total_release null : {int((ranged & releases['total_release'].isna()).sum()):,}")
print(f"  of those, release_na == '0'  : {int((ranged & (releases['release_na'] == '0')).sum()):,}")
print()
print(f"reported pounds                : {total_lb:,.0f}")
print(f"documented midpoints add       : {midpoint_lb:,.0f} lb"
      f" = {midpoint_lb / total_lb:.4%}")
print(f"even at every band maximum     : {band_high.sum():,.0f} lb"
      f" = {band_high.sum() / total_lb:.4%}")


range codes present:
release_range_code
1    56
3    26
4     2

range-coded records            : 84 of 2,100 = 4.00%
  of those, total_release null : 84
  of those, release_na == '0'  : 84

reported pounds                : 25,690,634
documented midpoints add       : 8,437 lb = 0.0328%
even at every band maximum     : 15,532 lb = 0.0605%


**4.00% of records and 0.0328% of pounds.** Even if every range-coded release
sat at the top of its band rather than the midpoint, the total would rise by 0.0605%.

Both halves of that matter. The loss is real, it is one-directional, and it is invisible in a
naive `sum()` — filtering to rows with a numeric `total_release` silently discards **84 of
2,100 applicable records, 4.00%**, and discards them non-randomly from the small end. And it
is negligible in mass, so a total-pounds figure is not meaningfully understated by it.
Whether 4% of records matters depends on the question: it is nothing for a mass total and it
is a real bias for anything that counts or ranks records.

It would have been easy to write the first half and stop. A directional loss sounds like it
should be material, and here it is not. The way to know is to compute the bound.

## 6. A vocabulary change inside one column

The basis-code scheme changed in reporting year 2007: `M` split into `M1` (continuous) and
`M2` (periodic), and `E` split into `E1` (published emission factors) and `E2` (site-specific).
A series spanning that year carries two vocabularies in one column.

The documented change year invites an obvious handling — old codes before 2007, new codes
after. The counts below test it. They are `count` queries only; no rows are downloaded.

In [8]:
"""Did the 2007 vocabulary change happen cleanly? Counts only -- no rows."""
from krl_tutorials_data.sources import verify_digest, _tri_count

PAIRS = {"monitoring": ("M", "M1", "M2"), "emission factors": ("E", "E1", "E2")}
rows = []
for year in SCHEME_YEARS:
    base = (f"tri_facility/state_abbr/{STATE}"
            f"/tri_reporting_form/reporting_year/{year}/tri_release_qty")
    row = {"year": year}
    for codes in PAIRS.values():
        for code in codes:
            row[code] = _tri_count(f"{base}/release_basis_est_code/{code}")
    rows.append(row)

scheme = pd.DataFrame(rows).set_index("year")
print(scheme.to_string(formatters={c: "{:,.0f}".format for c in scheme.columns}))

retired = ["M", "E"]
introduced = ["M1", "M2", "E1", "E2"]
print()
print(f"scheme change documented at reporting year {TRI_CODE_SCHEME_CHANGE_YEAR}")
for year in SCHEME_YEARS:
    old = int(scheme.loc[year, retired].sum())
    new = int(scheme.loc[year, introduced].sum())
    print(f"  {year}: retired-scheme records {old:>5,}   new-scheme records {new:>5,}")

#: What a code-only filter costs, printed rather than left as arithmetic the
#: reader has to do on the table above. The prose used to state these three
#: totals without the notebook ever computing them.
monitoring = ["M", "M1", "M2"]
all_monitoring = int(scheme[monitoring].to_numpy().sum())
only_m = int(scheme["M"].sum())
print()
print(f'monitoring-based records, {SCHEME_YEARS[0]}-{SCHEME_YEARS[-1]} : {all_monitoring:>5,}')
print(f'  caught by a `code == "M"` filter            : {only_m:>5,}')
print(f'  silently dropped by it                      : {all_monitoring - only_m:>5,}')


       M M1  M2   E  E1  E2
year                       
2006 437  0   2 963   3   0
2007   3 24 321  18 630 288
2008   3 28 336   7 716 275

scheme change documented at reporting year 2007
  2006: retired-scheme records 1,400   new-scheme records     5
  2007: retired-scheme records    21   new-scheme records 1,263
  2008: retired-scheme records    10   new-scheme records 1,355

monitoring-based records, 2006-2008 : 1,154
  caught by a `code == "M"` filter            :   443
  silently dropped by it                      :   711


The change is sharp but it is clean on neither side. Reporting year **2006**,
before the change, already carries **5** new-scheme records. Reporting year **2008**, a year
after it, still carries **10** retired-scheme records.

The consequence is concrete. Over 2006–2008 Virginia filed **1,154** monitoring-based release
records. A filter written as `code == "M"` — the documented pre-2007 monitoring code —
catches **443** of them and silently drops **711**. Filtering by year instead of by code has
the same problem in the opposite direction.

The correct handling is the one the client uses: treat `{M, M1, M2}` as a set, and never
infer the vocabulary from the year.

## 7. The transport hazard: a full page is not a full answer

Envirofacts caps a single response at 10,000 rows. It signals truncation in no way at all —
no flag, no next-page link, no partial-content status, no error. A query matching more than
the cap returns a well-formed, complete-looking JSON array of exactly 10,000 rows.

This is not a hypothetical.

In [9]:
"""The transport hazard: a full page is not a full answer."""
from krl_tutorials_data.sources import _tri_count

query = (f"tri_facility/state_abbr/{STATE}"
         f"/tri_reporting_form/reporting_year/{YEAR}/tri_release_qty")
declared = _tri_count(query)
one_page = min(declared, TRI_PAGE_LIMIT)

print(f"rows the query actually matches   : {declared:,}")
print(f"rows one request can return       : {TRI_PAGE_LIMIT:,} (Envirofacts cap)")
print(f"rows a single unpaged call returns: {one_page:,}")
print(f"silently missing from that call   : {1 - one_page / declared:.1%}")
print()
print("Nothing in that response says so. There is no truncation flag, no next-page")
print("link, and no error. The only signal available is the count endpoint, which")
print("is why this client calls it before it fetches a single row.")


rows the query actually matches   : 12,069
rows one request can return       : 10,000 (Envirofacts cap)
rows a single unpaged call returns: 10,000
silently missing from that call   : 17.1%

Nothing in that response says so. There is no truncation flag, no next-page
link, and no error. The only signal available is the count endpoint, which
is why this client calls it before it fetches a single row.


**17.1% of the answer**, missing, with nothing in the response to say so.

The only defence available is the `count` endpoint, which is why the client calls it before
it fetches a single row, pages against the declared total, and raises rather than returning a
frame whose length disagrees with it.

There is a second, quieter hazard in the same place. The service degrades sharply as it
approaches its own cap: measured against this query, 2,000 rows returned in 11.4 seconds and
5,000 in 21.0, while a 10,000-row request did not return inside 175 seconds. A client that
pages at the documented ceiling is a client that times out, so the request size is a separate
constant (`TRI_REQUEST_ROWS = 2,000`) sitting well below it.

## 8. Is any of this Virginia?

Every figure above is one state in one year. Three more states separate what belongs to the
regime from what belongs to Virginia.

In [10]:
"""Three more states: which of the above is the regime and which is Virginia?"""
comparison_rows = []
for abbrev in [STATE] + COMPARISON:
    if abbrev == STATE:
        frame, meta = releases, prov
    else:
        frame, meta = fetch_tri_releases(
            state=abbrev, reporting_year=YEAR, applicable_only=True
        )
    quant = frame[frame["total_release"].notna()]
    quant_measured = quant["is_measured"].fillna(False)
    pounds = quant["total_release"].sum()
    biggest = quant["total_release"].idxmax()
    without = quant.drop(index=biggest)
    without_measured = without["is_measured"].fillna(False)
    comparison_rows.append({
        "state": abbrev,
        "records": len(frame),
        "record measured": frame["is_measured"].fillna(False).mean(),
        "M1 share": (frame["release_basis_est_code"] == "M1").mean(),
        "mass measured": quant.loc[quant_measured, "total_release"].sum() / pounds,
        "mass, less top": without.loc[without_measured, "total_release"].sum()
                          / without["total_release"].sum(),
        "top record share": quant["total_release"].max() / pounds,
        "range records": frame["release_range_code"].notna().mean(),
    })

across = pd.DataFrame(comparison_rows).set_index("state")
print(across.to_string(formatters={
    "records": "{:,.0f}".format,
    "record measured": "{:.2%}".format, "M1 share": "{:.2%}".format,
    "mass measured": "{:.2%}".format, "mass, less top": "{:.2%}".format,
    "top record share": "{:.2%}".format, "range records": "{:.2%}".format,
}))
print()
print(f"record-weighted measured share, range across four states: "
      f"{across['record measured'].min():.2%} to {across['record measured'].max():.2%}")
print(f"mass-weighted   measured share, range across four states: "
      f"{across['mass measured'].min():.2%} to {across['mass measured'].max():.2%}")
print(f"M1 share stays inside                                   : "
      f"{across['M1 share'].min():.2%} to {across['M1 share'].max():.2%}")


      records record measured M1 share mass measured mass, less top top record share range records
state                                                                                             
VA      2,100          12.52%    0.76%        39.88%         12.22%           31.51%         4.00%
DE        297           5.72%    1.01%         7.69%         31.07%           75.23%         2.69%
RI        352           4.55%    1.14%        62.92%          4.48%           61.18%         5.68%
WV      2,166          18.47%    0.60%        11.71%         15.49%           24.40%         2.31%

record-weighted measured share, range across four states: 4.55% to 18.47%
mass-weighted   measured share, range across four states: 7.69% to 62.92%
M1 share stays inside                                   : 0.60% to 1.14%


The record-weighted monitored share ranges **4.55% to 18.47%** across the four
states — a real spread, and a narrow one next to the mass-weighted range of **7.69% to
62.92%**.

The instability is not merely wider. Its **direction is not consistent**. Dropping the single
largest record moves the mass-weighted share *down* in Virginia (39.88% → 12.22%) and Rhode
Island (62.92% → 4.48%), and *up* in Delaware (7.69% → 31.07%) and West Virginia (11.71% →
15.49%). A reader who checked one state and concluded "the top record inflates the monitored
share" would have the sign backwards in half the sample.

One figure is stable everywhere: **continuous monitoring (`M1`) is between 0.60% and 1.14% of
records in all four states.** Under one percent of published TRI release quantities rest on
continuous measurement, and that is a property of the programme, not of a state.

Range-coded records run 2.31% to 5.68% — the same order everywhere, and everywhere small.

## 9. What kind of source this is

TRI publishes no interval. There is no margin of error, no confidence bound, no suppression
threshold, no privacy parameter. Handbook §7.3 tracks a closed series on how much precision
is needed before a ranking is determined; **TRI cannot join that table**, because the
quantity it would need — an interval width — does not exist.

The absence of an interval is not the absence of uncertainty. It is the absence of a *stated*
uncertainty, and those are very different situations. What TRI gives instead is better in one
respect and worse in another.

**Better:** the uncertainty statement is per record. A margin of error tells you how well a
survey estimated a population. A basis code tells you how *this specific number* was arrived
at — and you can aggregate those codes any way the question requires, which is what sections
3 and 4 do. Almost nothing else in this corpus permits that.

**Worse:** the statement is categorical, not quantitative. "Best engineering judgment" does
not come with an error bar, and neither does "continuous monitoring data". You can say that
32.00% of records rest on judgment. You cannot say what that does to a total, because nothing
published tells you how far a judgment-based estimate typically sits from a measured one.

That is the regime, stated plainly: **epistemic provenance per record, with no quantification
attached to any of it.**

## Limitations

1. **The reporting universe is a threshold, not a sample.** Nothing here supports an inference
   about releases from facilities below the thresholds, and the number of such facilities is
   not published. Absence from TRI is not evidence of absence of release.

2. **The numbers are self-reported and unverified by EPA.** The basis code says what method
   the filer says they used. Nothing in the data attests that they used it.

3. **The basis code is categorical.** No quantification of error attaches to any code, so the
   record shares in section 2 cannot be converted into a bound on the total.

4. **The basis code is not a pure quality scale.** Section 2 shows `O` is used heavily to
   report zero. Treating the codes as an ordered quality ranking would conflate estimation
   method with reporting situation.

5. **Four states, one year.** Section 8 establishes that the record-weighted share and the
   `M1` share are stable across four states in 2022. It does not establish stability across
   all fifty, and it says nothing about stability over time.

6. **`M1` sub-analyses are underpowered.** Sixteen records, of which thirteen carry a
   quantity. The 46.2% zero rate quoted in section 2 rests on six observations and is
   reported only to forestall over-reading the table.

7. **The range-code midpoint is a reconstruction.** `range_midpoint_lb` implements what EPA's
   Data Dictionary says is stored. It is not a reported quantity and is never merged into
   `total_release`. If EPA's documentation and EPA's API disagree — and here they do — this
   notebook follows the documentation only for the separate column, and follows the API for
   everything else.

8. **Chemical identity is not used.** Pounds are summed across chemicals of wildly different
   toxicity. Every mass figure here is a mass figure and nothing more; none of it is a
   statement about harm.

**On vintage.** EPA Envirofacts serves current TRI data and facilities file corrections against past reporting years, so reporting_year names a reference period, not a vintage. This notebook therefore records `vintage_pinned: false` (handbook §5.2); the payload hash and retrieval time in its provenance record are what a later reader has to compare against.

## Key Takeaways

1. **TRI attaches epistemic provenance to every record.** Six basis-of-estimate codes
   distinguish continuous monitoring from periodic monitoring from mass balance from
   published and site-specific emission factors from best engineering judgment.

2. **Continuous monitoring is under 1% of records** — 0.60% to 1.14% across four states.
   Monitoring of any kind is 4.55% to 18.47%. The largest single category in Virginia is
   engineering judgment, at 32.00%.

3. **A mass-weighted quality statistic over a concentrated distribution is not a statistic.**
   Virginia's 39.88% falls to 12.22% when one of 2,016 records is removed, and the direction
   of that one-record effect is inconsistent across states.

4. **Test the prior rather than assuming it.** Measurement does not track release size here:
   ρ = +0.0280, p = 0.288, n = 1,439.

5. **Bound a directional loss in both currencies.** The range-code hole is 4.00% of records
   and 0.0328% of pounds. Directional does not imply material, and the only way to know is
   to compute it.

6. **A documented vocabulary change is not a clean cut.** Both schemes appear on both sides
   of reporting year 2007; filtering by the documented code drops 711 of 1,154 records.

7. **An API that caps a response and signals nothing requires a count-first client.** One
   unpaged call to this query returns a complete-looking 10,000 rows and omits 17.1%.

8. **No published interval is not no uncertainty.** TRI's uncertainty statement is per record
   and categorical, which permits aggregations no interval-based source permits, and supports
   no bound on a total.

## Exercises

1. **Re-weight by facility rather than by record or by pound.** A facility filing thirty Form
   R records contributes thirty rows to section 3. Compute the monitored share with each
   `doc_ctrl_num` — or each facility — weighted equally and say which of the three answers
   you would publish, and to whom.

2. **Find the leave-one-out threshold.** Section 3 drops the single largest record. Write a
   function that reports how many of the largest records must be removed before the
   mass-weighted monitored share settles inside ±1 percentage point of the record-weighted
   one, and run it for all four states. Does the required number track the top record's share
   of mass?

3. **Bound the range hole against a count question.** Section 5 shows the hole is negligible
   in pounds. Construct a question about Virginia 2022 for which the 4.00% of records *is*
   material — one where dropping the range-coded rows changes the answer by more than a few
   percent — and state it precisely.

4. **Test the scheme boundary elsewhere.** Section 6 uses Virginia. Run the same count
   queries for two other states and report whether the retired codes persist after 2007 at a
   similar rate. If they do, the residue is a programme property; if they do not, it is a
   state filing practice.

5. **Break the client and watch what happens.** Call `_tri_get` directly for this query with
   `rows/0:9999` and compare its length against `_tri_count`. Then write the one-line check
   that would have caught the problem, and say why a client cannot rely on the response alone.

6. **Argue the other side.** Section 3 prefers the record-weighted share. Construct the
   strongest case for the mass-weighted one — a decision for which it is the right statistic
   despite the instability — and state what you would need to publish alongside it.

## References

- U.S. Environmental Protection Agency. *Toxics Release Inventory (TRI) Program.*
  https://www.epa.gov/toxics-release-inventory-tri-program
- U.S. Environmental Protection Agency. *Envirofacts Data Service API.*
  https://www.epa.gov/enviro/envirofacts-data-service-api
- U.S. Environmental Protection Agency. *TRI Data Dictionary* — `RELEASE_BASIS_EST_CODE` and
  `RELEASE_RANGE_CODE` column definitions, served through Envirofacts column metadata at
  https://enviro.epa.gov/envirofacts/metadata/table/tri/tri_release_qty
- U.S. Environmental Protection Agency. *TRI Reporting Forms and Instructions (Form R).*
  https://www.epa.gov/toxics-release-inventory-tri-program/tri-reporting-forms-and-instructions
- Emergency Planning and Community Right-to-Know Act of 1986, 42 U.S.C. §11023 — the
  reporting thresholds that define the universe.
- Handbook §1.14 (this regime), §1.4 (suppression with a flag), §1.13 (voluntary reporting),
  §7.3 (the closed precision-and-rank series).